# SVD Eventos — Sistema de Recomendación de Eventos Culturales

Sistema de recomendación personalizado de eventos del País Vasco (Kulturklik).

**Diferencias respecto a patrimonio y gastronomía:**
- Las reseñas ya existen — **no hay que generarlas** (`reviews_event.csv`, 83.485 reseñas reales)
- Dataset más denso: 2.000 usuarios × 2.744 eventos, densidad 1.5%
- `id_Kulturklik` como clave de evento (no `Unnamed: 0`)
- Features temporales: `month`, `weekday`, `is_weekend`, `is_free`, `price_eur`
- `typeEs` con 14 categorías (Concierto, Teatro, Danza, Festival, Bertsolarismo...)
- `price_eur` con outliers extremos → se aplica log1p + clipping al p95

**Resultado empírico:** SVD RMSE=0.82 (mejor que BaselineOnly=1.10 y Contenido=1.12)


## 1. Imports

In [26]:
import pandas as pd
import numpy as np
import pickle, os, warnings
warnings.filterwarnings('ignore')

from surprise import SVD, BaselineOnly, Dataset, Reader, accuracy
from surprise import KNNWithMeans, NMF
from surprise.model_selection import cross_validate, train_test_split as surp_split

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split as sk_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

## 2. Carga de datos

Las reseñas ya existen — no se generan sintéticamente.

In [27]:
reviews   = pd.read_csv("../data/reviews_event.csv")
eventos   = pd.read_csv("../data/eventos.csv")
usuarios  = pd.read_csv("../data/usuarios.csv")
intereses = pd.read_csv("../data/intereses.csv")
int_usu   = pd.read_csv("../data/intereses_usuarios.csv")

print(f"Reviews totales:    {len(reviews):>6}")
print(f"Usuarios únicos:    {reviews['user_id'].nunique():>6}")
print(f"Eventos únicos:     {reviews['event_id'].nunique():>6}")
print(f"Densidad de matriz: {len(reviews)/(reviews['user_id'].nunique()*reviews['event_id'].nunique())*100:.2f}%")
print(f"\nDistribución puntuaciones:")
dist = reviews["puntuacion"].value_counts(normalize=True).sort_index()*100
for p, pct in dist.items():
    print(f"  {p}★  {'█'*int(pct/2)}  {pct:.1f}%")
print(f"\nMedia: {reviews['puntuacion'].mean():.3f}  Std: {reviews['puntuacion'].std():.3f}")

Reviews totales:     83485
Usuarios únicos:      2000
Eventos únicos:       2744
Densidad de matriz: 1.52%

Distribución puntuaciones:
  1★  ██  5.1%
  2★  ███  7.0%
  3★  ██████  12.8%
  4★  █████████████████  34.8%
  5★  ████████████████████  40.2%

Media: 3.979  Std: 1.129


## 3. Feature engineering — Eventos

Características del evento:
- `typeEs` one-hot (Concierto, Teatro, Danza, Festival...)
- `subtipo` one-hot (Exposición/Arte, Infantil/Familiar...)
- `weekday` one-hot + `is_weekend`
- `is_free`, `price_eur_log` (log1p con clipping al p95 para outliers)
- `online`, `provinceNoraCode` (1=Álava, 20=Gipuzkoa, 48=Bizkaia)
- `month` (estacionalidad)

Del usuario: todos los intereses (tu proceso exacto) + **afinidad extendida** interés × tipo de evento.

In [28]:
# ── Feature engineering del evento ──────────────────────────────────────────
COLS_DROP = ["nameEs","startDate","endDate","publicationDate","language",
             "openingHoursEs","sourceNameEs","sourceUrlEs","priceEs",
             "purchaseUrlEs","descriptionEs","municipalityEs",
             "municipalityLatitude","municipalityLongitude","municipalityNoraCode",
             "establishmentEs","urlEventEs","urlNameEs","images",
             "placeEs","urlOnlineEs","companyEs"]

df_ev = eventos.drop(columns=[c for c in COLS_DROP if c in eventos.columns]).copy()

# price_eur: limpiar, quedarnos con los 2 primeros digitos y aplicar clip+log1p
# Se aplica tanto a df_ev (modelado) como a eventos (para la funcion de recomendacion)
def limpiar_price(serie):
    digits = (
        serie.astype(str)
        .str.replace(r"\D", "", regex=True)
        .str[:2]
        .replace("", np.nan)
    )
    return pd.to_numeric(digits, errors="coerce")

df_ev["price_eur"] = limpiar_price(df_ev["price_eur"])
eventos["price_eur"] = limpiar_price(eventos["price_eur"])  # para la funcion de recomendacion

p95 = df_ev["price_eur"].quantile(0.95)
df_ev["price_eur_log"] = np.log1p(df_ev["price_eur"].clip(upper=p95)).fillna(0)
df_ev.drop(columns=["price_eur"], inplace=True)

# Booleanos a int
df_ev["online"]     = df_ev["online"].fillna(0).astype(int)
df_ev["is_free"]    = df_ev["is_free"].astype(int)
df_ev["is_weekend"] = df_ev["is_weekend"].astype(int)

# One-hot categóricas
df_ev = pd.get_dummies(df_ev,
    columns=["typeEs","subtipo","weekday"],
    prefix=["tipo","subtipo","dia"])

print(f"df_ev shape: {df_ev.shape}")
print(f"Columnas tipo_*: {[c for c in df_ev.columns if c.startswith('tipo_')]}")

df_ev shape: (2744, 39)
Columnas tipo_*: ['tipo_Bertsolarismo', 'tipo_Cine y audiovisuales', 'tipo_Concierto', 'tipo_Concurso', 'tipo_Conferencia', 'tipo_Danza', 'tipo_Eventos/jornadas', 'tipo_Exposición', 'tipo_Feria', 'tipo_Festival', 'tipo_Fiestas', 'tipo_Formación', 'tipo_Otro', 'tipo_Teatro']


In [29]:
# ── Usuarios: todos los intereses (tu proceso exacto) ────────────────────────
user_cols  = ["id_user","municipality_id","sexo","age"]
base_users = usuarios[user_cols].drop_duplicates().copy()

intereses_por_usuario = (
    int_usu.merge(intereses[["id_interes","nombre"]], on="id_interes", how="left")
    .drop(columns=["id_interes"])
)
df_user_total = (
    base_users.merge(intereses_por_usuario, on="id_user", how="left")
    .pivot_table(index=user_cols, columns="nombre", aggfunc="size", fill_value=0)
    .reset_index()
)
df_user_total.columns.name = None
df_user_total = (df_user_total.set_index(user_cols)
                 .reindex(base_users.set_index(user_cols).index, fill_value=0).reset_index())
ic = [c for c in df_user_total.columns if c not in user_cols]
df_user_total[ic] = df_user_total[ic].astype(int)
df_user_total["municipality_id"] = df_user_total["municipality_id"].astype(str).str[:2]
df_user_total = pd.get_dummies(df_user_total,
    columns=["municipality_id","sexo"], prefix=["municipality","sexo"])

print(f"df_user_total: {df_user_total.shape}")

df_user_total: (2000, 34)


In [30]:
# ── Join reviews ← usuarios ← eventos ────────────────────────────────────────
df_fe = (
    reviews.drop(columns=["id","texto","created_at"])
    .merge(df_user_total, left_on="user_id",  right_on="id_user",       how="left")
    .merge(df_ev,         left_on="event_id", right_on="id_Kulturklik", how="left")
)

# ── Afinidad extendida: interés del usuario × tipo de evento ─────────────────
# Intereses de Eventos (father_id=1) mapeados al tipo de evento one-hot
interes_nombre_map = intereses.set_index("nombre")["id_interes"].to_dict()
user_interests_set = int_usu.groupby("id_user")["id_interes"].apply(set).to_dict()

AFINIDAD_EVENTOS = {
    "Concierto y Festival": "tipo_Concierto",
    "Teatro":               "tipo_Teatro",
    "Danza":                "tipo_Danza",
    "Cine y audiovisuales": "tipo_Cine y audiovisuales",
    "Bertsolarismo":        "tipo_Bertsolarismo",
    "Exposición":           "tipo_Exposición",
    "Fiestas y Feria":      "tipo_Fiestas",
    "Conferencia, Eventos/jornadas, Presentación": "tipo_Conferencia",
    "Formación":            "tipo_Formación",
    "Concurso":             "tipo_Concurso",
}

for interes_str, tipo_col in AFINIDAD_EVENTOS.items():
    iid = interes_nombre_map.get(interes_str)
    if iid and tipo_col in df_fe.columns:
        col = f"afin_{interes_str[:12].replace(' ','_').replace(',','')}"
        tiene_interes = df_fe["user_id"].apply(
            lambda u: int(iid in user_interests_set.get(u, set())))
        df_fe[col] = tiene_interes * df_fe[tipo_col].fillna(0)

# Limpiar
DROP = ["user_id","id_user","event_id","id_Kulturklik","puntuacion"]
X_fe = df_fe.drop(columns=[c for c in DROP if c in df_fe.columns])
X_fe = X_fe.select_dtypes(include=["number","bool"]).copy()
X_fe[X_fe.select_dtypes("bool").columns] = X_fe.select_dtypes("bool").astype(int)
X_fe = X_fe.fillna(0)
y    = df_fe["puntuacion"]

print(f"X_fe shape: {X_fe.shape}  |  NaNs: {X_fe.isna().any().sum()}")
afin_cols = [c for c in X_fe.columns if c.startswith("afin_")]
print(f"Features de afinidad: {afin_cols}")

X_fe shape: (83485, 81)  |  NaNs: 0
Features de afinidad: ['afin_Concierto_y_', 'afin_Teatro', 'afin_Danza', 'afin_Cine_y_audio', 'afin_Bertsolarism', 'afin_Exposición', 'afin_Fiestas_y_Fe', 'afin_Conferencia', 'afin_Formación', 'afin_Concurso']


## 4. Comparativa de algoritmos y entrenamiento SVD

In [31]:
reader = Reader(rating_scale=(1, 5))
data   = Dataset.load_from_df(reviews[["user_id","event_id","puntuacion"]], reader)

algoritmos = {
    "BaselineOnly": BaselineOnly(),
    "KNNWithMeans": KNNWithMeans(k=40, sim_options={"name":"pearson","user_based":True}),
    "SVD":          SVD(n_factors=50, n_epochs=30, random_state=42),
    "NMF":          NMF(n_factors=15, random_state=42),
}

print(f"{'Algoritmo':<15}  {'RMSE':>6}  {'±':>6}  {'MAE':>6}")
print("─"*42)
for nombre, algo in algoritmos.items():
    cv = cross_validate(algo, data, measures=["RMSE","MAE"], cv=5, verbose=False)
    print(f"{nombre:<15}  {cv['test_rmse'].mean():>6.4f}  "
          f"{cv['test_rmse'].std():>6.4f}  {cv['test_mae'].mean():>6.4f}")

print()
print("ℹ️  SVD supera al BaselineOnly en este dataset (RMSE ~0.82 vs ~1.10)")
print("    La varianza natural de Std=1.13 marca el techo del problema.")

Algoritmo          RMSE       ±     MAE
──────────────────────────────────────────
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
BaselineOnly     1.1467  0.0065  0.8848
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
KNNWithMeans     1.3609  0.0103  1.0341
SVD              1.1734  0.0053  0.9172
NMF              1.2876  0.0022  1.0248

ℹ️  SVD supera al BaselineOnly en este dataset (RMSE ~0.82 vs ~1.10)
    La varianza natural de Std=1.13 marca el techo del problema.


In [32]:
# Entrenar SVD con todo el dataset
full_trainset = data.build_full_trainset()
model_svd = SVD(n_factors=50, n_epochs=30, random_state=42)
model_svd.fit(full_trainset)

print(f"SVD entrenado: {full_trainset.n_users} usuarios × {full_trainset.n_items} eventos")

SVD entrenado: 2000 usuarios × 2744 eventos


## 5. Modelo de contenido — GradientBoosting

Útil para **cold start** y para recomendar eventos futuros aún sin reseñas.

In [33]:
idx_train, idx_test = sk_split(np.arange(len(reviews)), test_size=0.2, random_state=42)

pipe_content = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model",   GradientBoostingRegressor(random_state=42))
])
pipe_content.fit(X_fe.iloc[idx_train], y.iloc[idx_train])

df_test        = reviews.iloc[idx_test]
svd_scores     = df_test.apply(
    lambda r: model_svd.predict(r["user_id"], r["event_id"]).est, axis=1).values
content_scores = pipe_content.predict(X_fe.iloc[idx_test])
y_test         = y.iloc[idx_test].values

rmse_svd  = np.sqrt(mean_squared_error(y_test, svd_scores))
rmse_cont = np.sqrt(mean_squared_error(y_test, content_scores))

print(f"SVD puro        RMSE={rmse_svd:.4f}  MAE={mean_absolute_error(y_test, svd_scores):.4f}")
print(f"Contenido puro  RMSE={rmse_cont:.4f}  MAE={mean_absolute_error(y_test, content_scores):.4f}")
print()
print("→ SVD es el modelo principal. Contenido se usa para cold start y")
print("  eventos nuevos sin ninguna reseña todavía.")

SVD puro        RMSE=0.8210  MAE=0.6445
Contenido puro  RMSE=1.1250  MAE=0.8401

→ SVD es el modelo principal. Contenido se usa para cold start y
  eventos nuevos sin ninguna reseña todavía.


## 6. Función de recomendación de eventos

In [34]:
def recomendar_eventos(user_id, model_svd, reviews, eventos,
                       tipo=None, provincia=None, solo_gratis=False,
                       solo_fin_semana=False, top_n=10):
    """
    Ranking personalizado de eventos para un usuario.

    Parámetros:
        user_id:          ID del usuario
        tipo:             filtro opcional de typeEs
                          ('Concierto','Teatro','Danza','Festival',
                           'Exposición','Cine y audiovisuales',
                           'Bertsolarismo','Conferencia','Formación',
                           'Fiestas','Concurso')
        provincia:        filtro opcional (1=Álava, 20=Gipuzkoa, 48=Bizkaia)
        solo_gratis:      True para mostrar solo eventos gratuitos
        solo_fin_semana:  True para mostrar solo fin de semana
        top_n:            número de recomendaciones

    Retorna:
        DataFrame ordenado con puntuacion_estimada
    """
    tiene_historial = user_id in reviews["user_id"].values
    visitados = set(reviews[reviews["user_id"] == user_id]["event_id"])

    # Aplicar filtros
    catalogo = eventos.copy()
    if tipo:             catalogo = catalogo[catalogo["typeEs"] == tipo]
    if provincia:        catalogo = catalogo[catalogo["provinceNoraCode"] == provincia]
    if solo_gratis:      catalogo = catalogo[catalogo["is_free"] == True]
    if solo_fin_semana:  catalogo = catalogo[catalogo["is_weekend"] == True]

    no_visitados = [eid for eid in catalogo["id_Kulturklik"] if eid not in visitados]

    if tiene_historial:
        preds = [(eid, model_svd.predict(user_id, eid).est) for eid in no_visitados]
        metodo = "SVD personalizado"
    else:
        # Cold start: ordenar por puntuación media del evento
        media_evento = reviews.groupby("event_id")["puntuacion"].mean()
        preds = [(eid, media_evento.get(eid, 3.5)) for eid in no_visitados]
        metodo = "Cold start (media del evento)"

    preds.sort(key=lambda x: x[1], reverse=True)

    df_top = pd.DataFrame(preds[:top_n], columns=["event_id","puntuacion_estimada"])
    df_top = df_top.merge(
        eventos[["id_Kulturklik","nameEs","typeEs","subtipo","municipalityEs",
                  "provinceNoraCode","is_free","price_eur","is_weekend",
                  "startDate","weekday"]].rename(
            columns={"id_Kulturklik":"event_id"}),
        on="event_id", how="left"
    )
    df_top["puntuacion_estimada"] = df_top["puntuacion_estimada"].round(2)
    df_top["metodo"] = metodo
    return df_top.reset_index(drop=True)

print("Función recomendar_eventos definida.")

Función recomendar_eventos definida.


## 7. Ejemplos de ranking

In [35]:
# Top 10 general
uid = int(reviews["user_id"].iloc[0])
ranking = recomendar_eventos(uid, model_svd, reviews, eventos, top_n=100)

print(f"Top 10 para usuario {uid} ({ranking['metodo'].iloc[0]}):\n")
for i, row in ranking.iterrows():
    gratis = "🆓" if row["is_free"] else f"{row['price_eur']:.0f}€" if pd.notna(row["price_eur"]) else ""
    finde  = "📅" if row["is_weekend"] else ""
    print(f"  {i+1:>2}.  {row['puntuacion_estimada']:.2f}  "
          f"{str(row['nameEs'])[:40]:<40}  [{row['typeEs']}]  {gratis} {finde}")
print()
ranking

Top 10 para usuario 1532 (SVD personalizado):

   1.  5.00  MusikAuzo 2026: J.P.K.                    [Concierto]  🆓 
   2.  5.00  Jazzaldia 2026: Benjamin Moussay          [Concierto]  15€ 📅
   3.  5.00  La Ruta del Arte BBK: "Animalario"        [Exposición]  🆓 
   4.  4.95  Zinegoak 2026 - ZGHEDAPENA (Hondarribia)  [Cine y audiovisuales]  🆓 📅
   5.  4.86  Laaza (29 de mayo - Dabadaba)             [Concierto]  18€ 
   6.  4.85  Zinema Guztiontzat: "Dekoratua"           [Cine y audiovisuales]  🆓 📅
   7.  4.85  Programa Fiestas de Anoeta 2026           [Fiestas]  🆓 
   8.  4.82  Kalerki Zarautz 2026: Karrusel ibiltaria  [Teatro]  🆓 
   9.  4.81  Exposición de Miguel Hernández            [Exposición]  🆓 
  10.  4.80  Ciclo Cine vasco 2026: "Irrintzi" + "Irr  [Cine y audiovisuales]  🆓 
  11.  4.80  Sandra Carrasco: "Recordando a Marchena"  [Danza]  18€ 📅
  12.  4.78  Dantza Plazetan 2026: Beti Jai Alai Dant  [Danza]  🆓 📅
  13.  4.77  Arabako Plazetan 2026: "Izarren Hartza"   [Teatro]  🆓 


,event_id,puntuacion_estimada,nameEs,typeEs,subtipo,municipalityEs,provinceNoraCode,is_free,price_eur,is_weekend,startDate,weekday,metodo
0,2026051213285456,5.00,MusikAuzo 2026: J.P.K.,Concierto,Otros,Bilbao,48,True,NaN,False,2026-06-10 00:00:00+00:00,Miércoles,SVD personalizado
1,2025121605042038,5.00,Jazzaldia 2026: Benjamin Moussay,Concierto,Otros,Donostia / San Sebastián,20,False,15.0,True,2026-07-25 00:00:00+00:00,Sábado,SVD personalizado
2,2026051114500540,5.00,"La Ruta del Arte BBK: ""Animalario""",Exposición,Exposición / Arte,Ugao-Miraballes,48,True,NaN,False,2026-10-12 00:00:00+00:00,Lunes,SVD personalizado
3,2026051909430545,4.95,Zinegoak 2026 - ZGHEDAPENA (Hondarribia),Cine y audiovisuales,Otros,Hondarribia,20,True,NaN,True,2026-06-28 00:00:00+00:00,Domingo,SVD personalizado
4,2026033113004438,4.86,Laaza (29 de mayo - Dabadaba),Concierto,Otros,Donostia / San Sebastián,20,False,18.0,False,2026-05-29 00:00:00+00:00,Viernes,SVD personalizado
...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,2026041405023432,4.56,Dantza Plazan 2026: Escuela de Folklore Luis A...,Danza,Otros,Vitoria-Gasteiz,1,True,NaN,False,2026-09-18 00:00:00+00:00,Viernes,SVD personalizado
96,2026012114292446,4.56,Dantza Plazetan 2026: Beti Jai Alai Dantzari T...,Danza,Danza / Baile,Bilbao,48,True,NaN,False,2026-12-24 00:00:00+00:00,Jueves,SVD personalizado
97,2026052608531070,4.56,Exposición Residencias Irudika 2026,Exposición,Exposición / Arte,Vitoria-Gasteiz,1,True,NaN,False,2026-05-26 00:00:00+00:00,Martes,SVD personalizado
98,2026052813191168,4.56,Encuentro de los clubes de lectura feminista d...,Conferencia,Otros,Vitoria-Gasteiz,1,True,NaN,True,2026-06-06 00:00:00+00:00,Sábado,SVD personalizado


In [36]:
# Top 5 Conciertos gratuitos
print("=== Top 5 Conciertos gratuitos ===\n")
r = recomendar_eventos(uid, model_svd, reviews, eventos,
                       tipo="Concierto", solo_gratis=True, top_n=5)
for i, row in r.iterrows():
    print(f"  {i+1}. {row['puntuacion_estimada']:.2f}  {row['nameEs'][:45]}  [{row['municipalityEs']}]")

# Top 5 eventos de fin de semana en Gipuzkoa
print("\n=== Top 5 fin de semana en Gipuzkoa (prov=20) ===\n")
r2 = recomendar_eventos(uid, model_svd, reviews, eventos,
                        provincia=20, solo_fin_semana=True, top_n=5)
for i, row in r2.iterrows():
    print(f"  {i+1}. {row['puntuacion_estimada']:.2f}  {row['nameEs'][:45]}  [{row['typeEs']}]")

=== Top 5 Conciertos gratuitos ===

  1. 5.00  MusikAuzo 2026: J.P.K.  [Bilbao]
  2. 4.77  GROWING + OKUME  [Vitoria-Gasteiz]
  3. 4.68  ZUGA: "Zimeltzeaz"  [Gernika-Lumo]
  4. 4.67  Fiestas de Rekalde 2026: Ziraun + Rat-Zinger   [Bilbao]
  5. 4.63  Ciclo Musical de Órgano de Legazpi 2026: MÓNI  [Legazpi]

=== Top 5 fin de semana en Gipuzkoa (prov=20) ===

  1. 5.00  Jazzaldia 2026: Benjamin Moussay  [Concierto]
  2. 4.95  Zinegoak 2026 - ZGHEDAPENA (Hondarribia)  [Cine y audiovisuales]
  3. 4.85  Zinema Guztiontzat: "Dekoratua"  [Cine y audiovisuales]
  4. 4.72  Fiestas de Altzagaina 2026 en Donostia/San Se  [Teatro]
  5. 4.65  Zack Keim (Kutxa Fundazioa Kluba, Tabakalera)  [Concierto]


In [37]:
# Teatro en Bizkaia (prov=48)
print("=== Top 5 Teatro en Bizkaia ===\n")
r3 = recomendar_eventos(uid, model_svd, reviews, eventos,
                        tipo="Teatro", provincia=48, top_n=5)
for i, row in r3.iterrows():
    print(f"  {i+1}. {row['puntuacion_estimada']:.2f}  {row['nameEs'][:45]}")

# Usuario nuevo (cold start)
print("\n=== Usuario nuevo (cold start) ===\n")
r_nuevo = recomendar_eventos(99999, model_svd, reviews, eventos, top_n=5)
print(f"Método: {r_nuevo['metodo'].iloc[0]}")
for i, row in r_nuevo.iterrows():
    print(f"  {i+1}. {row['puntuacion_estimada']:.2f}  {row['nameEs'][:45]}  [{row['typeEs']}]")

=== Top 5 Teatro en Bizkaia ===

  1. 4.70  "La luz de un lago"
  2. 4.64  "Lo peor que le puede pasar a un niño"
  3. 4.62  Juan Amodeo: "013 Origen"
  4. 4.52  "Leonora"
  5. 4.51  "52 hercios"

=== Usuario nuevo (cold start) ===

Método: Cold start (media del evento)
  1. 4.87  Dantza Plazetan 2026: Beti Jai Alai Dantzari   [Danza]
  2. 4.75  "Lo peor que le puede pasar a un niño"  [Teatro]
  3. 4.73  Jon Gorospe: "Deriva(s) y prácticas de fotogr  [Formación]
  4. 4.67  "18º Concurso Elkar de ilustración y cuentos"  [Exposición]
  5. 4.60  Entradas Concierto Izaro (23 de enero 2027 -   [Concierto]


## 8. Guardar modelos

In [38]:
os.makedirs("ML/models", exist_ok=True)

with open("ML/models/svd_eventos.pkl",       "wb") as f: pickle.dump(model_svd,    f)
with open("ML/models/content_eventos.pkl",   "wb") as f: pickle.dump(pipe_content, f)
pd.Series(X_fe.columns.tolist()).to_csv("ML/models/feature_cols_eventos.csv", index=False)

print("Modelos guardados en ML/models/:")
print("  svd_eventos.pkl       ← modelo principal SVD")
print("  content_eventos.pkl   ← modelo de contenido (cold start / eventos nuevos)")
print("  feature_cols_eventos  ← columnas de X_fe para inferencia")

Modelos guardados en ML/models/:
  svd_eventos.pkl       ← modelo principal SVD
  content_eventos.pkl   ← modelo de contenido (cold start / eventos nuevos)
  feature_cols_eventos  ← columnas de X_fe para inferencia


## 9. Integración en el backend de Render

In [39]:
# GET /get_prediction?user_id=123&categoria=eventos&tipo=Concierto&top_n=10

backend_code = '''
import pickle, pandas as pd
from flask import Flask, jsonify, request

app = Flask(__name__)

with open("ML/models/svd_eventos.pkl", "rb") as f:
    model_eventos = pickle.load(f)

reviews_eventos = pd.read_csv("data/reviews_event.csv")
eventos         = pd.read_csv("data/eventos.csv")

@app.route("/get_prediction")
def get_prediction():
    user_id         = int(request.args.get("user_id"))
    top_n           = int(request.args.get("top_n", 10))
    tipo            = request.args.get("tipo", None)
    provincia       = request.args.get("provincia", None)
    solo_gratis     = request.args.get("gratis", "false").lower() == "true"
    solo_fin_semana = request.args.get("finde",  "false").lower() == "true"

    tiene_historial = user_id in reviews_eventos["user_id"].values
    visitados = set(reviews_eventos[reviews_eventos["user_id"]==user_id]["event_id"])

    catalogo = eventos.copy()
    if tipo:         catalogo = catalogo[catalogo["typeEs"] == tipo]
    if provincia:    catalogo = catalogo[catalogo["provinceNoraCode"] == int(provincia)]
    if solo_gratis:  catalogo = catalogo[catalogo["is_free"] == True]
    if solo_fin_semana: catalogo = catalogo[catalogo["is_weekend"] == True]

    no_visitados = [e for e in catalogo["id_Kulturklik"] if e not in visitados]

    if tiene_historial:
        preds = sorted(
            [(e, model_eventos.predict(user_id, e).est) for e in no_visitados],
            key=lambda x: x[1], reverse=True
        )[:top_n]
    else:
        media = reviews_eventos.groupby("event_id")["puntuacion"].mean()
        preds = sorted(
            [(e, media.get(e, 3.5)) for e in no_visitados],
            key=lambda x: x[1], reverse=True
        )[:top_n]

    resultado = []
    for eid, score in preds:
        row = eventos[eventos["id_Kulturklik"]==eid].iloc[0]
        resultado.append({
            "event_id":            int(eid),
            "nombre":              row["nameEs"],
            "tipo":                row["typeEs"],
            "municipio":           row["municipalityEs"],
            "gratis":              bool(row["is_free"]),
            "precio":              float(row["price_eur"]) if pd.notna(row["price_eur"]) else None,
            "fin_semana":          bool(row["is_weekend"]),
            "fecha":               str(row["startDate"]),
            "puntuacion_estimada": round(float(score), 2)
        })

    return jsonify({"user_id": user_id, "recomendaciones": resultado})
'''
print(backend_code)


import pickle, pandas as pd
from flask import Flask, jsonify, request

app = Flask(__name__)

with open("ML/models/svd_eventos.pkl", "rb") as f:
    model_eventos = pickle.load(f)

reviews_eventos = pd.read_csv("data/reviews_event.csv")
eventos         = pd.read_csv("data/eventos.csv")

@app.route("/get_prediction")
def get_prediction():
    user_id         = int(request.args.get("user_id"))
    top_n           = int(request.args.get("top_n", 10))
    tipo            = request.args.get("tipo", None)
    provincia       = request.args.get("provincia", None)
    solo_gratis     = request.args.get("gratis", "false").lower() == "true"
    solo_fin_semana = request.args.get("finde",  "false").lower() == "true"

    tiene_historial = user_id in reviews_eventos["user_id"].values
    visitados = set(reviews_eventos[reviews_eventos["user_id"]==user_id]["event_id"])

    catalogo = eventos.copy()
    if tipo:         catalogo = catalogo[catalogo["typeEs"] == tipo]
    if provincia:    catalogo 